# COMP9517 2026 T2 — Lab 4

## Image segmentation with Otsu thresholding and morphological processing

This notebook implements a **single algorithm** that segments objects in the lab images using:

1. Grayscale conversion
2. **Otsu** automatic intensity thresholding
3. Binary **opening** and **closing**
4. **Morphological reconstruction** to remove objects touching the image boundary
5. Removal of small connected components
6. Counting and displaying the remaining objects

The same functions are applied to all images; only parameter values differ per image.


## Step 1: Import relevant packages

We use **OpenCV** for image I/O, thresholding, and morphological operations, and **matplotlib** for visualisation.


In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

print(f"OpenCV version: {cv2.__version__}")


## Step 2: Segmentation algorithm

The functions below implement the complete pipeline described in the lab specification and Week 5 lecture slides (Part 2).

User-adjustable parameters are exposed as function arguments:

- `open_kernel_size` / `close_kernel_size` — morphological filter sizes (0 disables that step)
- `min_area` — minimum connected-component area to keep
- `invert` — whether to use inverted Otsu binarisation
- `connectivity` — 4 or 8 for connected-component analysis


In [ ]:
def morphological_reconstruction(marker: np.ndarray, mask: np.ndarray, kernel=None) -> np.ndarray:
    """Geodesic dilation of marker constrained by mask until convergence."""
    if kernel is None:
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    current = marker.copy()
    while True:
        dilated = cv2.dilate(current, kernel, iterations=1)
        new = cv2.bitwise_and(dilated, mask)
        if np.array_equal(new, current):
            break
        current = new
    return current


def remove_border_objects(binary: np.ndarray, connectivity: int = 8) -> np.ndarray:
    """Remove foreground components that touch the image boundary via reconstruction."""
    h, w = binary.shape
    border = np.zeros((h, w), dtype=np.uint8)
    border[0, :] = 255
    border[-1, :] = 255
    border[:, 0] = 255
    border[:, -1] = 255

    seeds = cv2.bitwise_and(binary, border)
    border_region = morphological_reconstruction(seeds, binary)
    return cv2.subtract(binary, border_region)


def remove_small_components(binary: np.ndarray, min_area: int, connectivity: int = 8) -> np.ndarray:
    """Remove connected components smaller than min_area pixels."""
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity)
    cleaned = np.zeros_like(binary)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == label] = 255
    return cleaned


def count_objects(binary: np.ndarray, connectivity: int = 8) -> int:
    """Count foreground connected components (background label excluded)."""
    num_labels, _, _, _ = cv2.connectedComponentsWithStats(binary, connectivity)
    return num_labels - 1


def segment_and_count_objects(
    image: np.ndarray,
    open_kernel_size: int = 3,
    close_kernel_size: int = 5,
    min_area: int = 100,
    invert: bool = False,
    connectivity: int = 8,
    blur_ksize: int = 5,
):
    """Segment an image and return the final mask, object count, and intermediate results."""
    if image.ndim == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()

    if blur_ksize and blur_ksize >= 3:
        k = blur_ksize if blur_ksize % 2 == 1 else blur_ksize + 1
        gray_blur = cv2.GaussianBlur(gray, (k, k), 0)
    else:
        gray_blur = gray

    thresh_type = cv2.THRESH_BINARY_INV if invert else cv2.THRESH_BINARY
    _, otsu_binary = cv2.threshold(gray_blur, 0, 255, thresh_type | cv2.THRESH_OTSU)

    binary = otsu_binary
    if open_kernel_size and open_kernel_size > 0:
        k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel_size, open_kernel_size))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k_open)
    if close_kernel_size and close_kernel_size > 0:
        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel_size, close_kernel_size))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close)

    after_morph = binary.copy()
    no_border = remove_border_objects(binary, connectivity=connectivity)
    final_mask = remove_small_components(no_border, min_area=min_area, connectivity=connectivity)
    n_objects = count_objects(final_mask, connectivity=connectivity)

    intermediates = {
        "gray": gray,
        "otsu_binary": otsu_binary,
        "after_morph": after_morph,
        "no_border": no_border,
    }
    return final_mask, n_objects, intermediates


## Step 3: Load images and set parameters

Place the lab images in `COMP9517_26T2_Lab4_Images/` next to this notebook (or update `IMAGE_DIR` below).

Expected stems: `Fruits`, `Pineapples`, `Monsters` (`.jpg`, `.png`, `.bmp`, or `.tif`).

The parameter dictionary holds the **only** per-image differences in the pipeline.


In [ ]:
# Directory containing the three lab images (relative to notebook location)
IMAGE_DIR = Path("COMP9517_26T2_Lab4_Images")

# Uncomment and edit if you prefer an absolute path on your machine:
# IMAGE_DIR = Path(r"C:\Users\11928\Desktop\9517\lab04\COMP9517_26T2_Lab4_Images")

IMAGE_STEMS = ["Fruits", "Pineapples", "Monsters"]
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]


def resolve_image_path(image_dir: Path, stem: str) -> Path:
    for ext in IMAGE_EXTENSIONS:
        candidate = image_dir / f"{stem}{ext}"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find image for '{stem}' in {image_dir}. "
        f"Tried: {[stem + ext for ext in IMAGE_EXTENSIONS]}"
    )


# Per-image parameters (same algorithm, different values)
IMAGE_PARAMS = {
    "Fruits": {
        "open_kernel_size": 5,
        "close_kernel_size": 7,
        "min_area": 800,
        "invert": False,
        "connectivity": 8,
        "blur_ksize": 5,
    },
    "Pineapples": {
        "open_kernel_size": 5,
        "close_kernel_size": 9,
        "min_area": 1200,
        "invert": False,
        "connectivity": 8,
        "blur_ksize": 5,
    },
    "Monsters": {
        "open_kernel_size": 3,
        "close_kernel_size": 5,
        "min_area": 600,
        "invert": True,
        "connectivity": 8,
        "blur_ksize": 5,
    },
}


## Step 4: Run segmentation on all images

For each image we display the original, grayscale, Otsu mask, mask after opening/closing, and the final segmentation with the object count.


### Expected results (from the lab specification)

| Image | Expected count |
|-------|----------------|
| Fruits | n = 19 |
| Pineapples | n = 15 |
| Monsters | n = 18 |

If your counts differ, adjust values in `IMAGE_PARAMS` (especially `min_area`, kernel sizes, and `invert`).

In [ ]:
results = {}

for stem in IMAGE_STEMS:
    image_path = resolve_image_path(IMAGE_DIR, stem)
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"Failed to read image: {image_path}")

    params = IMAGE_PARAMS[stem]
    final_mask, n_objects, intermediates = segment_and_count_objects(image_bgr, **params)

    results[stem] = {
        "path": image_path,
        "image_bgr": image_bgr,
        "final_mask": final_mask,
        "n_objects": n_objects,
        "intermediates": intermediates,
        "params": params,
    }

    print(f"{stem}: n = {n_objects}")


## Step 5: Visualise results

Side-by-side summary matching the layout in the lab specification.


In [ ]:
fig, axes = plt.subplots(len(IMAGE_STEMS), 5, figsize=(18, 4 * len(IMAGE_STEMS)))
if len(IMAGE_STEMS) == 1:
    axes = np.expand_dims(axes, axis=0)

column_titles = ["Original", "Grayscale", "Otsu", "After open/close", "Final segmentation"]

for row, stem in enumerate(IMAGE_STEMS):
    r = results[stem]
    image_rgb = cv2.cvtColor(r["image_bgr"], cv2.COLOR_BGR2RGB)
    gray = r["intermediates"]["gray"]
    otsu = r["intermediates"]["otsu_binary"]
    morph = r["intermediates"]["after_morph"]
    final_mask = r["final_mask"]

    row_images = [image_rgb, gray, otsu, morph, final_mask]
    for col, (ax, img, title) in enumerate(zip(axes[row], row_images, column_titles)):
        if col == 0:
            ax.imshow(img)
        else:
            ax.imshow(img, cmap="gray")
        if row == 0:
            ax.set_title(title)
        if col == 4:
            ax.set_xlabel(f"n = {r['n_objects']}", fontsize=12)
        ax.axis("off")

    axes[row, 0].set_ylabel(stem, fontsize=12, rotation=0, labelpad=60, va="center")

plt.suptitle("Lab 4 segmentation results", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Compact summary figure (as in the lab specification examples)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, stem in zip(axes, IMAGE_STEMS):
    r = results[stem]
    ax.imshow(r["final_mask"], cmap="gray")
    ax.set_title(stem)
    ax.set_xlabel(f"n = {r['n_objects']}")
    ax.axis("off")

plt.suptitle("Final segmentation masks", fontsize=14)
plt.tight_layout()
plt.show()
